# H3b: Partial Singular-Value Equalization

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H3b_PARTIAL_SV_EQUALIZATION/run_experiment.py`

## Notebook purpose

This notebook is the academically clearer counterpart to the script. It **imports and uses the script's implementation directly** rather than re-defining the training core.

## Truthful scope of the current experiment

This is a **toy deep-linear final-loss benchmark**. It tests whether partially flattening the momentum-buffer singular-value spectrum helps final optimization relative to SGD with momentum and a full-polar (`k=d`) Muon-style reference.

What is measured here:
- final loss after training,
- per-method LR sweep behavior,
- gap-closure summaries relative to SGD and the full-polar reference,
- explicit T1/T2/T3 checks.

What is **not** measured here:
- per-step singular-value rescue during training,
- update effective rank / subspace collapse,
- any direct weak-direction-boosting metric.

So any mechanistic interpretation below should be read as **consistent with** the final-loss evidence in this toy setting, not as a direct measurement of singular-value dynamics.


In [ ]:
from __future__ import annotations

import importlib.util
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

EXPERIMENT_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/H3b_PARTIAL_SV_EQUALIZATION')


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / EXPERIMENT_RELATIVE / 'run_experiment.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate the repository root by searching upward from '
        f'{start}. Expected to find {EXPERIMENT_RELATIVE / "run_experiment.py"}.'
    )


REPO_ROOT = find_repo_root(Path.cwd())
EXPERIMENT_DIR = REPO_ROOT / EXPERIMENT_RELATIVE
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('h3b_partial_sv_equalization', SCRIPT_PATH)
h3b = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h3b)

config = h3b.get_experiment_config()

print('Repository root:', REPO_ROOT)
print('Experiment directory:', EXPERIMENT_DIR)
print('Notebook path:', NOTEBOOK_PATH)
print('Script path:', SCRIPT_PATH)
print('CLI reproduction command:')
print('  ', config['cli_reproduction_command'])


## Reproducibility and runtime

The script fixes the full default toy setup:
- 4-layer deep linear network,
- matrix dimension `d = 32`,
- `500` training steps,
- momentum `0.9`,
- `5` seeds total,
- `12` LR candidates,
- sweep over `k in {1, 2, 4, 8, 16, 32}`.

The LR sweep uses the first three seeds for model-selection-style LR choice, then evaluates on all five seeds.

Because the sweep is repeated independently for each `k` plus SGD, the default run is not instantaneous. The script exposes the expected training-run budget so the notebook can report it honestly.


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None


def show_table(rows, columns=None, title=None, index=False):
    if title:
        print(title)
    if not rows:
        print('(empty)')
        return None
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.to_string(index=index))
        return df

    if columns is None:
        columns = list(rows[0].keys())
    widths = {col: max(len(str(col)), max(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(f'{col:<{widths[col]}}' for col in columns)
    divider = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(' | '.join(f'{str(row.get(col, "")):<{widths[col]}}' for col in columns))
    return rows


config_rows = [
    {'field': 'experiment_id', 'value': config['experiment_id']},
    {'field': 'counterpart_script', 'value': config['counterpart_script']},
    {'field': 'counterpart_notebook', 'value': config['counterpart_notebook']},
    {'field': 'dim', 'value': config['dim']},
    {'field': 'num_layers', 'value': config['num_layers']},
    {'field': 'num_steps', 'value': config['num_steps']},
    {'field': 'momentum', 'value': config['momentum']},
    {'field': 'batch_size', 'value': config['batch_size']},
    {'field': 'seeds', 'value': config['seeds']},
    {'field': 'selection_seeds', 'value': config['selection_seeds']},
    {'field': 'k_values', 'value': config['k_values']},
    {'field': 'lr_candidates', 'value': config['lr_candidates']},
    {'field': 'expected_training_runs', 'value': config['expected_training_runs']},
    {'field': 'target_step_frobenius_norm', 'value': config['target_step_frobenius_norm']},
]
show_table(config_rows, columns=['field', 'value'], title='Default experiment configuration')


## Precisely what the implemented transform does

For a momentum matrix
\[
M = U \operatorname{diag}(\sigma) V^T,
\]
with `sigma` returned by SVD in descending order, the implementation does **exactly** this:

1. Take the **first `k` entries of that input spectrum** and replace them by their mean.
2. Keep the remaining tail entries proportional to their original values.
3. Rescale the resulting spectrum so the reconstructed matrix has Frobenius norm `sqrt(d)`, matching the full polar-factor norm.
4. Reconstruct the matrix as `U diag(sigma_new) V^T`.

Important precision points:
- The notebook/script do **not** measure singular-value dynamics over training; they only use this transform inside the optimizer and then report final loss.
- The statement "top-`k` singular values are equal" should be read as referring to the **input SVD slots before reconstruction**.
- The tail is **not** unchanged; it is rescaled in a `k`-dependent way.
- Because the LR is also tuned independently for each `k`, the experiment varies both the spectral transform and the best-performing LR chosen for that transform.
- `k = d` should exactly match the polar factor `U V^T`; this is checked explicitly below as T3.


In [ ]:
rng = np.random.RandomState(999)
M_demo = rng.randn(config['dim'], config['dim'])
_, sigma_demo, _ = np.linalg.svd(M_demo, full_matrices=False)

k_demo_values = [1, 4, 16, config['dim']]
transform_rows = []
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(sigma_demo) + 1), sigma_demo, marker='o', linewidth=2, label='original spectrum')

for k in k_demo_values:
    sigma_new = h3b.transform_spectrum(sigma_demo, k)
    transform_rows.append({
        'k': k,
        'input_top_mean': float(np.mean(sigma_demo[:k])),
        'output_fro_norm': float(np.linalg.norm(sigma_new)),
        'first_entry': float(sigma_new[0]),
        'kth_entry': float(sigma_new[k - 1]),
        'tail_last_entry': float(sigma_new[-1]),
    })
    plt.plot(np.arange(1, len(sigma_new) + 1), sigma_new, marker='.', linewidth=1.5, label=f'k={k}')

plt.axhline(np.sqrt(config['dim'] / config['dim']), color='gray', alpha=0.0)
plt.xlabel('Spectrum index')
plt.ylabel('Singular value after transform')
plt.title('Implemented spectral transform on one random matrix')
plt.legend()
plt.tight_layout()
plt.show()

show_table(
    transform_rows,
    columns=['k', 'input_top_mean', 'output_fro_norm', 'first_entry', 'kth_entry', 'tail_last_entry'],
    title='Diagnostic rows for the implemented spectrum transform',
)

M_full = h3b.partial_sv_equalize(M_demo, config['dim'])
M_polar = h3b.polar_factor(M_demo)
print('Full-polar matrix check on the demo matrix:')
print('  ||partial_sv_equalize(M, d) - polar_factor(M)||_F =', np.linalg.norm(M_full - M_polar))


## Execute the default script experiment from the notebook

This cell calls the script's `run_experiment()` function directly. That keeps the notebook aligned with the script and avoids duplicate training logic.

The returned result dictionary includes:
- config and seeds,
- LR sweep records,
- chosen best LRs,
- per-seed final losses,
- mean/std summaries,
- derived gap-closure metrics,
- explicit T1/T2/T3 checks,
- runtime metadata.


In [ ]:
run_start = time.time()
results = h3b.run_experiment(verbose=False, save_plot=False)
notebook_elapsed = time.time() - run_start

print(f'Notebook call to run_experiment finished in {notebook_elapsed:.1f} s.')
print(f"Script-reported runtime: {results['runtime_sec']:.1f} s")
print(f"Expected training runs: {results['config']['expected_training_runs']}")
print('Best LR choices:')
for k in results['config']['k_values']:
    print(f"  k={k:>2}: {results['best_lrs']['partial'][k]:.4f}")
print(f"  SGD: {results['best_lrs']['sgd']:.4f}")


## Summary tables

The first table summarizes the LR sweep winners. The second table summarizes the final evaluation. Negative `% gap closed` values are retained as-is because they indicate settings that performed **worse than SGD** under the benchmark's own normalization.


In [ ]:
lr_rows = []
for k in results['config']['k_values']:
    sweep = results['lr_sweep']['partial'][k]
    lr_rows.append({
        'label': f'k={k}',
        'best_lr': sweep['best_lr'],
        'selection_mean_loss': sweep['best_selection_mean_loss'],
        'n_selection_seeds_finite': sweep['records'][results['config']['lr_candidates'].index(sweep['best_lr'])]['n_finite'],
    })

lr_rows.append({
    'label': 'SGD',
    'best_lr': results['lr_sweep']['sgd']['best_lr'],
    'selection_mean_loss': results['lr_sweep']['sgd']['best_selection_mean_loss'],
    'n_selection_seeds_finite': results['lr_sweep']['sgd']['records'][results['config']['lr_candidates'].index(results['lr_sweep']['sgd']['best_lr'])]['n_finite'],
})

show_table(
    lr_rows,
    columns=['label', 'best_lr', 'selection_mean_loss', 'n_selection_seeds_finite'],
    title='Best-LR summary table',
)

summary_rows = []
for row in results['summary_rows']:
    summary_rows.append({
        'label': row['label'],
        'best_lr': row['best_lr'],
        'mean_loss': row['mean_loss'],
        'std_loss': row['std_loss'],
        'gap_closed_pct': row['gap_closed_pct'],
        'vs_sgd': row['vs_sgd'],
        'vs_full_polar': row['vs_full_polar'],
        'n_diverged': row['n_diverged'],
    })

show_table(
    summary_rows,
    columns=['label', 'best_lr', 'mean_loss', 'std_loss', 'gap_closed_pct', 'vs_sgd', 'vs_full_polar', 'n_diverged'],
    title='Final evaluation summary',
)


## Final mean loss vs `k`

This figure reports the quantity the current toy experiment actually measures most directly: **mean final loss** after training, with seed-to-seed standard deviation as the uncertainty bar.

Interpretation discipline:
- Lower is better.
- Non-monotonic behavior is possible and should be reported honestly.
- This plot does **not** directly reveal why a given `k` wins or loses; it only reports end-state performance.


In [ ]:
k_vals = results['config']['k_values']
loss_means = [results['evaluation']['partial'][k]['mean_loss'] for k in k_vals]
loss_stds = [results['evaluation']['partial'][k]['std_loss'] for k in k_vals]
sgd_mean = results['evaluation']['sgd']['mean_loss']
full_polar_mean = results['derived_metrics']['full_polar_mean_loss']

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(k_vals, loss_means, yerr=loss_stds, marker='o', linewidth=2, capsize=4, color='#CC3311', label='partial SV equalization')
ax.axhline(sgd_mean, color='#4477AA', linestyle='--', linewidth=1.5, label=f'SGD ({sgd_mean:.2e})')
ax.axhline(full_polar_mean, color='#228B22', linestyle='--', linewidth=1.5, label=f'k={config["dim"]} full polar ({full_polar_mean:.2e})')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xticks(k_vals)
ax.set_xticklabels([str(k) for k in k_vals])
ax.set_xlabel('k (number of input-spectrum entries flattened)')
ax.set_ylabel('Final loss')
ax.set_title('Final mean loss vs k')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

is_monotone = all(loss_means[i] <= loss_means[i - 1] for i in range(1, len(loss_means)))
print('Monotone decrease in mean loss as k increases?', is_monotone)
print('Best-performing partial setting:', f"k={k_vals[int(np.argmin(loss_means))]}")


## `%` gap closed vs `k`

The normalization here is
\[
100 	imes \frac{L_{\mathrm{SGD}} - L_k}{L_{\mathrm{SGD}} - L_{\mathrm{full\ polar}}}.
\]

Interpretation:
- `100%` means matching the full-polar reference.
- `0%` means no improvement over SGD.
- **Negative values mean worse than SGD**.

Because negative values are scientifically important in this benchmark, the axis is not clipped to a positive-only range.


In [ ]:
gap_vals = [results['derived_metrics']['by_k'][k]['gap_closed_pct'] for k in k_vals]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(k_vals, gap_vals, marker='s', linewidth=2, color='#9933CC')
ax.axhline(0, color='black', linewidth=1, alpha=0.5)
ax.axhline(50, color='gray', linestyle=':', alpha=0.7, label='T1 threshold (50%)')
ax.axhline(80, color='gray', linestyle='--', alpha=0.7, label='T2 threshold (80%)')
ax.axhline(100, color='#228B22', linestyle='-', alpha=0.35, label='full polar = 100%')
low = min(gap_vals + [0.0])
high = max(gap_vals + [100.0])
pad = max(5.0, 0.08 * (high - low if high > low else 1.0))
ax.set_ylim(low - pad, high + pad)
ax.set_xscale('log', base=2)
ax.set_xticks(k_vals)
ax.set_xticklabels([str(k) for k in k_vals])
ax.set_xlabel('k (number of input-spectrum entries flattened)')
ax.set_ylabel('% of SGD→full-polar gap closed')
ax.set_title('Gap closure vs k')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Gap-closure values by k:')
for k, gap in zip(k_vals, gap_vals):
    status = 'worse than SGD' if gap < 0 else 'better than or equal to SGD'
    print(f'  k={k:>2}: {gap:>8.1f}%   ({status})')


## T1 / T2 / T3 checks

The current pair-level benchmark defines three explicit checks:

- **T1:** whether `k=1` captures more than `50%` of the SGD→full-polar gap.
- **T2:** whether there is an early knee, defined here as `>80%` gap closure at `k < d/2`.
- **T3:** whether the `k=d` transform implements the **explicit full polar factor** at the matrix level, with same-LR training-loss comparisons retained only as a numerical diagnostic.

T3 is the main correctness/sanity test for the implemented optimizer transform.


In [ ]:
test_rows = []
for key in ['T1', 'T2', 'T3']:
    payload = results['tests'][key]
    row = {'test': key, 'passed': payload['passed'], 'description': payload['description']}
    if key == 'T1':
        row['primary_value'] = payload['gap_closed_pct']
    elif key == 'T2':
        row['primary_value'] = payload['knee_gap_closed_pct'] if payload['knee_k'] is not None else np.nan
        row['knee_k'] = payload['knee_k']
    elif key == 'T3':
        row['primary_value'] = payload['matrix_check']['max_frobenius_error']
        row['max_loss_diff_diagnostic'] = payload['max_abs_loss_diff']
    test_rows.append(row)

show_table(test_rows, title='T1/T2/T3 summary')

print('Detailed T3 reference check:')
print('  reference LR:', results['tests']['T3']['reference_lr'])
print('  k=d losses:        ', results['tests']['T3']['k_dim_losses'])
print('  explicit ref losses:', results['tests']['T3']['reference_losses'])
print('  max |loss diff| diagnostic:   ', results['tests']['T3']['max_abs_loss_diff'])
print('  mean |loss diff| diagnostic:  ', results['tests']['T3']['mean_abs_loss_diff'])
print('  max matrix Frobenius error:', results['tests']['T3']['matrix_check']['max_frobenius_error'])
print('  note:', results['tests']['T3']['diagnostic_note'])


## Raw per-seed final losses

A serious notebook should make the seed-level end-state data inspectable rather than only showing aggregated means.


In [ ]:
seed_rows = []
for seed_idx, seed in enumerate(results['seeds']):
    row = {'seed': seed, 'SGD': results['evaluation']['sgd']['final_losses'][seed_idx]}
    for k in k_vals:
        row[f'k={k}'] = results['evaluation']['partial'][k]['final_losses'][seed_idx]
    row['full_polar_reference'] = results['evaluation']['full_polar_reference']['final_losses'][seed_idx]
    seed_rows.append(row)

show_table(seed_rows, title='Per-seed final losses')


## Calibrated conclusion

This conclusion should answer the pair-level question directly: **did partial SV equalization recover Muon-like behavior in the current default run?**

The wording below is intentionally restrained so that it matches the actual metrics in the script/notebook pair.


In [ ]:
t1 = results['tests']['T1']['passed']
t2 = results['tests']['T2']['passed']
t3 = results['tests']['T3']['passed']
full_polar_partial_best = np.argmin(loss_means) == len(loss_means) - 1
full_polar_beats_sgd = full_polar_mean < sgd_mean
any_partial_beats_sgd = any(results['evaluation']['partial'][k]['mean_loss'] < sgd_mean for k in k_vals if k != config['dim'])

print('Conclusion for the current default run:')
if full_polar_partial_best and full_polar_beats_sgd and not any_partial_beats_sgd:
    print(
        '  In this toy benchmark, partial SV equalization did NOT recover Muon-like behavior. '
        'Only the full-polar endpoint k=d beat SGD and clearly matched the best observed behavior, and '
        'the smaller-k settings did not outperform SGD.'
    )
elif full_polar_partial_best and full_polar_beats_sgd:
    print(
        '  In this toy benchmark, some partial settings helped, but the strongest behavior '
        'still concentrated at the full-polar endpoint k=d rather than being fully recovered '
        'at small k.'
    )
elif full_polar_beats_sgd:
    print(
        '  In this toy benchmark, the full-polar reference beat SGD, but it was not uniquely best among '
        'the partial settings. The current final-loss evidence therefore does not support a simple '
        '"only full equalization works" story.'
    )
else:
    print(
        '  In this toy benchmark, even the full-polar endpoint did not clearly beat SGD, so the current '
        'final-loss evidence does not support Muon-like recovery under this setup.'
    )

print('\nT1/T2/T3 verdicts:')
print(f'  T1 passed? {t1}')
print(f'  T2 passed? {t2}')
print(f'  T3 passed? {t3}')
print('\nInterpretation discipline:')
print(
    '  These results are evidence about end-state optimization performance under this toy setup. '
    'They are not a direct measurement of singular-value rescue over training, so stronger mechanistic '
    'claims would require additional diagnostics.'
)


## Optional full three-panel script figure

For completeness, the script also exposes a reusable plotting helper. The dedicated figures above are the primary notebook presentation; this panel reproduces the script-style overview inline.


In [ ]:
fig, axes = h3b.plot_results(results, save_path=None, show=False)
plt.show()
